In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from datasets import load_dataset
import pandas as pd
from PIL import Image
from tqdm import tqdm


In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 1
LEARNING_RATE = 0.001
TRAIN_SAMPLES = 1000
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, hf_dataset, label, transform=None):
        self.dataset = hf_dataset
        self.label = label
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image = self.dataset[idx]['image']
        if image.mode != 'RGB':
            image = image.convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor([self.label], dtype=torch.float32)


In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

real_ds = load_dataset("bitmind/ffhq-256_training_faces", split=f'train[:{TRAIN_SAMPLES}]')
fake_ds = load_dataset("bitmind/ffhq-256_stable-diffusion-xl-base-1.0_training_faces", split=f'train[:{TRAIN_SAMPLES}]')

train_dataset = torch.utils.data.ConcatDataset([
    DeepfakeDataset(real_ds, label=0, transform=transform),
    DeepfakeDataset(fake_ds, label=1, transform=transform)
])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0) [cite: 36]


In [ ]:
model = models.resnet18(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 1)
model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)


In [ ]:
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Loss: {running_loss/len(train_loader):.4f}")

test_dir = '/content/test_images' 
submission = pd.read_csv('sample_submission.csv') # 기존 양식 로드 [cite: 41]


In [ ]:
model.eval()
scores = []

with torch.no_grad():
    for img_id in tqdm(submission['id'], desc="Predicting"):
        img_path = os.path.join(test_dir, img_id)
        image = Image.open(img_path).convert('RGB')
        input_tensor = transform(image).unsqueeze(0).to(DEVICE)
        
        logit = model(input_tensor)
        prob = torch.sigmoid(logit).item() # Fake일 확률 계산 [cite: 24, 40]
        scores.append(prob)

submission['score'] = scores
submission.to_csv('submission_baseline.csv', index=False)